In [0]:
%run "../notebooks/helper_functions"

In [0]:
# Create widgets for configuration
dbutils.widgets.text("catalog_name", "novacart_catalog", "1. Target Catalog Name")
dbutils.widgets.text("foreign_catalog_name", "novacart_external_mssql_oltp_db", "2. Source Catalog Name")
dbutils.widgets.text("bronze_schema", "bronze_schema", "3. Bronze Schema Name")
dbutils.widgets.text("source_schema", "dbo", "4. Source Schema Name")
dbutils.widgets.text("job_run_id", "", "5. Job Run ID from Databricks Workflow Job")

# Read widget values
catalog_name = dbutils.widgets.get("catalog_name").strip()
foreign_catalog_name = dbutils.widgets.get("foreign_catalog_name").strip()
bronze_schema = dbutils.widgets.get("bronze_schema").strip()
source_schema = dbutils.widgets.get("source_schema").strip()
job_run_id = dbutils.widgets.get("job_run_id").strip()

# Construct table prefixes and control table name
source_catalog_prefix = f"{foreign_catalog_name}.{source_schema}"
target_catalog_prefix = f"{catalog_name}.{bronze_schema}"
bronze_control_table = f"{catalog_name}.{bronze_schema}.ingestion_control"

print("=" * 60)
print("Bronze Configuration:")
print("=" * 60)
print(f"Source Catalog:        {foreign_catalog_name}")
print(f"Source Schema:         {source_schema}")
print(f"Target Catalog:        {catalog_name}")
print(f"Bronze Schema:         {bronze_schema}")
print("=" * 60)
print(f"Source Prefix:         {source_catalog_prefix}")
print(f"Target Prefix:         {target_catalog_prefix}")
print(f"Bronze Control Table:  {bronze_control_table}")
print("=" * 60)

#### Source table configuration
This cell defines which source tables will be loaded into Bronze and which columns should be used as:
- **primary key**
- **timestamp / watermark colum**
It also creates a unique **bronze_run_id** for the current pipeline run.

In [0]:
import uuid

In [0]:
tables_config = {
  "orders" : {"pk_col": "order_id", "ts_col": "updated_at"},
  "products" : {"pk_col": "product_id", "ts_col": "updated_at"},
  "payments" : {"pk_col": "payment_id", "ts_col": "processed_at"},
}

if job_run_id not in ["", None]:
  bronze_run_id = job_run_id
else:
  # Not running as a job, generate a UUID
  bronze_run_id = str(uuid.uuid4())
  print(f"Running interactively, generated UUID: {bronze_run_id}")

print(f"Current Bronze Run ID is: {bronze_run_id}")

#### Bronze incremental load loop
This is the main Bronze logic
For each table, the notebook:
1. reads the last watermark
2. reads the source SQL table
3. filters only new/changed rows
4. adds Bronze audit columns
5. appends the rows into the Bronze Delta table
6. update the control table

This is the core incremental loading logic

In [0]:
for table_name, cfg in tables_config.items():
  pk_col = cfg.get("pk_col")
  ts_col = cfg.get("ts_col")
  source_table = f"{source_catalog_prefix}.{table_name}"
  target_table = f"{target_catalog_prefix}.{table_name}_raw"
  
  # Pass control_table parameter
  last_successful_ts, last_successful_pk = get_last_successfull_watermark(table_name, bronze_control_table)

  # Fix - Truncate watermark to millisecond precision
  if last_successful_ts is not None:
    last_successful_ts = last_successful_ts.replace(
      microsecond=(last_successful_ts.microsecond // 1000) * 1000
    )

  print(f"\n*** Processing {table_name} ***")
  print(f"Last successful ts: {last_successful_ts}")
  print(f"Last successful pk: {last_successful_pk}")

  source_df = spark.read.table(source_table) \
    .withColumn(ts_col, F.date_trunc("MILLISECOND", F.col(ts_col).cast("timestamp")))
  
  if last_successful_ts is None:
    rows_to_load_df = source_df
  else:
    if last_successful_pk is None:
      rows_to_load_df = source_df.filter(
        F.col(ts_col) > F.lit(last_successful_ts)
      )
    else:
      rows_to_load_df = source_df.filter(
        (F.col(ts_col) > F.lit(last_successful_ts)) |
        (
          (F.col(ts_col) == F.lit(last_successful_ts)) &
          (F.col(pk_col).cast("long") > F.lit(last_successful_pk))
        )
      )

  rows_to_load_df = (
    rows_to_load_df
    .withColumn("bronze_ingested_at", F.current_timestamp())
    .withColumn("bronze_run_id", F.lit(bronze_run_id))
    .withColumn("bronze_source_table", F.lit(source_table))
  )

  rows_count = rows_to_load_df.count()
  print(f"{table_name} rows_to_load = {rows_count}")

  if rows_count == 0:
    print(f"No new rows to load for {table_name}.")
    # Pass control_table parameter
    upsert_bronze_control(
      table_name,
      ts_col,
      pk_col,
      last_successful_ts,
      last_successful_pk,
      rows_count,
      bronze_run_id,
      bronze_control_table
    )
    continue
  
  rows_to_load_df.write.format("delta").mode("append").saveAsTable(target_table)

  watermark_row = (
    rows_to_load_df.select(ts_col, pk_col)
    .orderBy(F.col(ts_col).desc(), F.col(pk_col).cast("long").desc())
    .limit(1)
    .collect()[0]
  )

  max_ts = watermark_row[ts_col]
  max_pk = int(watermark_row[pk_col]) if watermark_row[pk_col] is not None else None

  # Pass control_table parameter
  upsert_bronze_control(
      table_name,
      ts_col,
      pk_col,
      max_ts,
      max_pk,
      rows_count,
      bronze_run_id,
      bronze_control_table
    )
  
  print(f"Wrote {rows_count} to {target_table}")
  

#### Quick validation

In [0]:
print("Orders Bronze count:", spark.sql(f"select count(*) from {target_catalog_prefix}.orders_raw").collect()[0][0])

print("Products Bronze count:", spark.sql(f"select count(*) from {target_catalog_prefix}.products_raw").collect()[0][0])

print("Payments Bronze count:", spark.sql(f"select count(*) from {target_catalog_prefix}.payments_raw").collect()[0][0])


display(spark.sql(f"select * from {bronze_control_table}").orderBy("table_name"))